# 03 ? Panel model selection and fixed-effects benchmark

## tl;dr

Using the same observed-data sample throughout (6,911 country-years, 130 countries, 60 years), the project compares pooled OLS, random effects, and fixed effects while retaining a full set of year effects. Pooled OLS is decisively rejected, and the Hausman diagnostic rejects random effects (p < 0.001). The two-way fixed-effects model is therefore the primary association benchmark.


## Context & methods

### Key assumptions

- Outcome: observed PPP GDP per capita in natural logs.
- Primary regressors: private capital per capita, government capital per capita, and effective labor per capita, all in natural logs. Effective labor follows the original project: labor force ? normalized HCI.
- The pooled and random-effects models include an intercept and year indicators. The fixed-effects model includes country and year effects. There is **no separate linear time trend**.
- The intercept is a baseline level, not a measure of total factor productivity. PWT's constructed TFP series is held for a separate growth-accounting extension rather than included in this primary association model.
- Standard errors are clustered by country. The conventional Hausman diagnostic is reported separately because it uses its conventional covariance form.

The workflow is: pooled OLS with year effects; an F test for country effects; random effects with year effects; an unbalanced-panel variance-component LM test; the Hausman diagnostic; and then the two-way fixed-effects benchmark.


In [ ]:
### 1. Load the observed-data panel
import pandas as pd
from src.econometrics import build_baseline_log_sample, fit_panel_model_sequence

panel = pd.read_csv("data/processed/analysis_panel.csv")
estimation_sample = build_baseline_log_sample(panel)
print(
    f"Common sample: {len(estimation_sample):,} country-years; "
    f"{estimation_sample.country_code.nunique():,} countries; "
    f"{estimation_sample.year.nunique():,} years"
)


In [ ]:
### 2. Estimate pooled OLS, random effects, and fixed effects
coefficients, diagnostics, summary = fit_panel_model_sequence(estimation_sample)
coefficients.to_csv("results/tables/panel_model_coefficients.csv", index=False)
diagnostics.to_csv("results/tables/panel_model_diagnostics.csv", index=False)
summary.to_csv("results/tables/panel_model_summary.csv", index=False)
summary


In [ ]:
### 3. Evaluate the specification tests
# Pooled OLS is tested against FE and RE; the Hausman test compares RE and FE.
diagnostics


In [ ]:
### 4. Inspect the fixed-effects benchmark
fixed_effects = coefficients.query("model == 'Fixed effects + year effects'")
fixed_effects


## Technology extension

Technology is analyzed as a separate extension because researcher coverage is shorter than the core sample. This avoids silently changing the main estimation sample.


In [ ]:
### 5. Run the technology extension on its own observed-data sample
technology_sample = build_baseline_log_sample(panel, include_technology=True)
tech_coefficients, tech_diagnostics, tech_summary = fit_panel_model_sequence(
    technology_sample, include_technology=True
)
tech_coefficients.to_csv("results/tables/technology_extension_coefficients.csv", index=False)
tech_diagnostics.to_csv("results/tables/technology_extension_diagnostics.csv", index=False)
tech_summary.to_csv("results/tables/technology_extension_summary.csv", index=False)
print(
    f"Technology-extension sample: {len(technology_sample):,} country-years; "
    f"{technology_sample.country_code.nunique():,} countries"
)
tech_coefficients.query("model == 'Fixed effects + year effects'")


## Takeaways

- Full year effects capture common global shocks and changes in the average outcome each year; they are not a linear time trend.
- Country effects are strongly supported relative to pooled OLS, and the Hausman diagnostic rejects random effects in this sample.
- The fixed-effects results are association estimates, not causal effects: simultaneity between income and capital remains a central limitation and motivates later dynamic-panel work.
- Technology remains important substantively, but it is added in a separately reported extension so data coverage is transparent.
